<h4>Building a Deep Neural Network </h4>

here we will learn to implement all the function to build deep neural network



**Notation**:
- Superscript $[l]$ denotes a quantity associated with the $l^{th}$ layer. 
    - Example: $a^{[L]}$ is the $L^{th}$ layer activation. $W^{[L]}$ and $b^{[L]}$ are the $L^{th}$ layer parameters.
- Superscript $(i)$ denotes a quantity associated with the $i^{th}$ example. 
    - Example: $x^{(i)}$ is the $i^{th}$ training example.
- Lowerscript $i$ denotes the $i^{th}$ entry of a vector.
    - Example: $a^{[l]}_i$ denotes the $i^{th}$ entry of the $l^{th}$ layer's activations).

In [8]:
import numpy as np
import matplotlib.pyplot as plt
from dnn_utils import sigmoid, sigmoid_backward, relu ,relu_backward

import copy
%matplotlib inline
plt.rcParams['figure.figsize'] = (5.0, 4.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

%load_ext autoreload
%autoreload 2

np.random.seed(1)


<h4>heres a detail to about how to build a neural netwrok </h4>

1. initialize the parameters for a two layer and L-Layer neural network i.e is W parameter and b
2. Now time to implement forward propagations
   1. compute the LINEAR part of layer's forward propagation reuslt = $Z^[l]$
   2. ACTIVATION function that we are using is (relu/sigmoid) result $A^l = g(Z^l)$
   3. Combine step 1 & 2 into a new one LINEAR ACTIVATION forward function
   4. Now see what happen is till L-1 layers we are doing LINEAR->RELU function and for L_th layer meaning last layer we are using SIGMOID for prediction LINEAR -> SIGMOID
   5. This whole result in L_model_forward_function
3. Now Forward layer done and prediction make now time to <b>compute loss</b> so that we will get to know how bad our model has performed
4. <b>Implement backpropagation </b> to mininze loss
    1. Compute linear part of layer;s backward propagation step
    2. ACTIVATION function use in backward is <b>sigmoid_backward</b>, <b>relu_backward</b> that find derivates
    3. Linear backward dW,db, dA gives gradient that in which direction weights should change
    4. Except last layer till L-1 layer use ReLu backward and at Lth layer use Sigmoid backward
5. Now at last update the parameters
    


![screenshot](final_outline.png)

- every forward step has its backward step
- what ever calculation done in forward  it has its backward and <B>reverse + derivative</b>
- <b>Cache values </b> when ever forward propagation pas some value store
- that Z, A, W kind
- Why we need this while finding gradient during backward we need this value to find derivaties
- Thats why cache value savd
  

<h3>Initialization of parameters </h3>

<h4>2-Layer Neural Netwrok</h4>

<h5>Initialize the parameters</h5>

- create and initalize the parameters fo 2-layers neural network
- LINEAR -> RELU -> LINEAR -> SIGMOID

In [14]:
#initializing parameters
def initialize_parameters(n_x, n_h, n_y):
    """
    Arguments
    n_x -- size of input layers
    n_h -- size of hidden layer
    n_y -- size of output layers

    Return:
    parameters -- python dictionary that contains parameters
    w1(n_h, n_x)
    b1(n_h, 1)
    w2(n_y, n_h),  b2(n_y, 1)
    """
    np.random.seed(1)

#initialize the parameters

    W1 = np.random.randn(n_h, n_x) * 0.01
    b1 = np.zeros((n_h, 1)) # bias are define as zero vector
    W2 = np.random.randn(n_y, n_h) * 0.01
    b2 = np.zeros((n_y, 1))

#define (parameters) -  dictornary to store w1, b1, w2, b2 values
    parameters = {"W1": W1,
              "b1": b1,
              "W2": W2,
              "b2": b2}
    return parameters

In [15]:
parameters = initialize_parameters(3,2,1)
print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

W1 = [[ 0.01624345 -0.00611756 -0.00528172]
 [-0.01072969  0.00865408 -0.02301539]]
b1 = [[0.]
 [0.]]
W2 = [[ 0.01744812 -0.00761207]]
b2 = [[0.]]


<h3>L-layer Neural Network</h3>
for large neural networks it get complicated as there are several numbers of weight matrics and bias vectors initialize as zeros<br>

- it is important that dimension must match between each layers

<table style="width:100%">
    <tr>
        <td>  </td> 
        <td> <b>Shape of W</b> </td> 
        <td> <b>Shape of b</b>  </td> 
        <td> <b>Activation</b> </td>
        <td> <b>Shape of Activation</b> </td> 
    <tr>
    <tr>
        <td> <b>Layer 1</b> </td> 
        <td> $(n^{[1]},12288)$ </td> 
        <td> $(n^{[1]},1)$ </td> 
        <td> $Z^{[1]} = W^{[1]}  X + b^{[1]} $ </td> 
        <td> $(n^{[1]},209)$ </td> 
    <tr>
    <tr>
        <td> <b>Layer 2</b> </td> 
        <td> $(n^{[2]}, n^{[1]})$  </td> 
        <td> $(n^{[2]},1)$ </td> 
        <td>$Z^{[2]} = W^{[2]} A^{[1]} + b^{[2]}$ </td> 
        <td> $(n^{[2]}, 209)$ </td> 
    <tr>
       <tr>
        <td> $\vdots$ </td> 
        <td> $\vdots$  </td> 
        <td> $\vdots$  </td> 
        <td> $\vdots$</td> 
        <td> $\vdots$  </td> 
    <tr>  
   <tr>
       <td> <b>Layer L-1</b> </td> 
        <td> $(n^{[L-1]}, n^{[L-2]})$ </td> 
        <td> $(n^{[L-1]}, 1)$  </td> 
        <td>$Z^{[L-1]} =  W^{[L-1]} A^{[L-2]} + b^{[L-1]}$ </td> 
        <td> $(n^{[L-1]}, 209)$ </td> 
   <tr>
   <tr>
       <td> <b>Layer L</b> </td> 
        <td> $(n^{[L]}, n^{[L-1]})$ </td> 
        <td> $(n^{[L]}, 1)$ </td>
        <td> $Z^{[L]} =  W^{[L]} A^{[L-1]} + b^{[L]}$</td>
        <td> $(n^{[L]}, 209)$  </td> 
    <tr>
</table>

<a name='3-2'></a>
### 3.2 - L-layer Neural Network

The initialization for a deeper L-layer neural network is more complicated because there are many more weight matrices and bias vectors. When completing the `initialize_parameters_deep` function, you should make sure that your dimensions match between each layer. Recall that $n^{[l]}$ is the number of units in layer $l$. For example, if the size of your input $X$ is $(12288, 209)$ (with $m=209$ examples) then:

<table style="width:100%">
    <tr>
        <td>  </td> 
        <td> <b>Shape of W</b> </td> 
        <td> <b>Shape of b</b>  </td> 
        <td> <b>Activation</b> </td>
        <td> <b>Shape of Activation</b> </td> 
    <tr>
    <tr>
        <td> <b>Layer 1</b> </td> 
        <td> $(n^{[1]},12288)$ </td> 
        <td> $(n^{[1]},1)$ </td> 
        <td> $Z^{[1]} = W^{[1]}  X + b^{[1]} $ </td> 
        <td> $(n^{[1]},209)$ </td> 
    <tr>
    <tr>
        <td> <b>Layer 2</b> </td> 
        <td> $(n^{[2]}, n^{[1]})$  </td> 
        <td> $(n^{[2]},1)$ </td> 
        <td>$Z^{[2]} = W^{[2]} A^{[1]} + b^{[2]}$ </td> 
        <td> $(n^{[2]}, 209)$ </td> 
    <tr>
       <tr>
        <td> $\vdots$ </td> 
        <td> $\vdots$  </td> 
        <td> $\vdots$  </td> 
        <td> $\vdots$</td> 
        <td> $\vdots$  </td> 
    <tr>  
   <tr>
       <td> <b>Layer L-1</b> </td> 
        <td> $(n^{[L-1]}, n^{[L-2]})$ </td> 
        <td> $(n^{[L-1]}, 1)$  </td> 
        <td>$Z^{[L-1]} =  W^{[L-1]} A^{[L-2]} + b^{[L-1]}$ </td> 
        <td> $(n^{[L-1]}, 209)$ </td> 
   <tr>
   <tr>
       <td> <b>Layer L</b> </td> 
        <td> $(n^{[L]}, n^{[L-1]})$ </td> 
        <td> $(n^{[L]}, 1)$ </td>
        <td> $Z^{[L]} =  W^{[L]} A^{[L-1]} + b^{[L]}$</td>
        <td> $(n^{[L]}, 209)$  </td> 
    <tr>
</table>

Remember that when you compute $W X + b$ in python, it carries out broadcasting. For example, if: 

$$ W = \begin{bmatrix}
    w_{00}  & w_{01} & w_{02} \\
    w_{10}  & w_{11} & w_{12} \\
    w_{20}  & w_{21} & w_{22} 
\end{bmatrix}\;\;\; X = \begin{bmatrix}
    x_{00}  & x_{01} & x_{02} \\
    x_{10}  & x_{11} & x_{12} \\
    x_{20}  & x_{21} & x_{22} 
\end{bmatrix} \;\;\; b =\begin{bmatrix}
    b_0  \\
    b_1  \\
    b_2
\end{bmatrix}\tag{2}$$

Then $WX + b$ will be:

$$ WX + b = \begin{bmatrix}
    (w_{00}x_{00} + w_{01}x_{10} + w_{02}x_{20}) + b_0 & (w_{00}x_{01} + w_{01}x_{11} + w_{02}x_{21}) + b_0 & \cdots \\
    (w_{10}x_{00} + w_{11}x_{10} + w_{12}x_{20}) + b_1 & (w_{10}x_{01} + w_{11}x_{11} + w_{12}x_{21}) + b_1 & \cdots \\
    (w_{20}x_{00} + w_{21}x_{10} + w_{22}x_{20}) + b_2 &  (w_{20}x_{01} + w_{21}x_{11} + w_{22}x_{21}) + b_2 & \cdots
\end{bmatrix}\tag{3}  $$


In [25]:
#Now building deep neural networks 
def initialize_parameters_deep(layer_dims):
    """
    Arguments:
        layer_dims = it is python array(list) containig the dimensions of each layers in neural netwrok

    Return:
        parametes = python dictorinary containing parameters "w1", "b1", "wL, "bl"
                    wl -- weight matrix of shape(layer_dims[l], layer_dims[l-1])
                    bl -- bias vector of shape(layer_dims[l], 1)
    """

    np.random.seed(3)
    parameters = {}
    L = len(layer_dims) #number of layers in the network

    for l in range(1, L):
        parameters['W' +str(l)] = np.random.randn(layer_dims[l], layer_dims[l-1]) * 0.01
        parameters['b' +str(l)] = np.zeros((layer_dims[l], 1))

    return parameters
    

In [29]:
parameters = initialize_parameters_deep([5,4,3])
# these numbers are number of neurons in each layer here only three are defined
print("W1 = " + str(parameters["W1"]))
print("b1 = " + str(parameters["b1"]))
print("W2 = " + str(parameters["W2"]))
print("b2 = " + str(parameters["b2"]))

W1 = [[ 0.01788628  0.0043651   0.00096497 -0.01863493 -0.00277388]
 [-0.00354759 -0.00082741 -0.00627001 -0.00043818 -0.00477218]
 [-0.01313865  0.00884622  0.00881318  0.01709573  0.00050034]
 [-0.00404677 -0.0054536  -0.01546477  0.00982367 -0.01101068]]
b1 = [[0.]
 [0.]
 [0.]
 [0.]]
W2 = [[-0.01185047 -0.0020565   0.01486148  0.00236716]
 [-0.01023785 -0.00712993  0.00625245 -0.00160513]
 [-0.00768836 -0.00230031  0.00745056  0.01976111]]
b2 = [[0.]
 [0.]
 [0.]]


<h4>Forward propagation Module </h4>

<h5>Linear Forward</h5>
<br>
linear -> activation function(relu) -> (l-1)sigmoid

In [33]:
def linear_forward(A, W, b):
    """
    implementing linear part of a layers forward propagation

    Arguments:
    A -- activation from previous layer (size of previous layer, no od examples)
    W -- weight matrix (size of current layer, size of previous layers)
    b -- bias vectore ( size of current layer , 1)

    Returns:
    Z -- pre - activation parameters
    cache -- a python tuple that contain A, W, b stored for computing backward pass effiecently
    """
#linear function
    Z = np.dot(A, W) + b

#storing cache files 
    cache = (A, W, b)

    return Z, cache


<h4>Linear- Activation Forward</h4>
<a name='4-2'></a>
### 4.2 - Linear-Activation Forward

In this notebook, you will use two activation functions:

- **Sigmoid**: $\sigma(Z) = \sigma(W A + b) = \frac{1}{ 1 + e^{-(W A + b)}}$. You've been provided with the `sigmoid` function which returns **two** items: the activation value "`a`" and a "`cache`" that contains "`Z`" (it's what we will feed in to the corresponding backward function). To use it you could just call: 
``` python
A, activation_cache = sigmoid(Z)
```

- **ReLU**: The mathematical formula for ReLu is $A = RELU(Z) = max(0, Z)$. You've been provided with the `relu` function. This function returns **two** items: the activation value "`A`" and a "`cache`" that contains "`Z`" (it's what you'll feed in to the corresponding backward function). To use it you could just call:
``` python
A, activation_cache = relu(Z)
```

In [44]:
#now grouping linear function and activation function into one 
def linear_activation_forward(A_prev, W, b, activation):
    """
    implementing forward propagationa as linear-> activation layer

    Arguments:
    A_prev -- activation from prevoius later : (size of previous layer, number of examples)
    W -- weights matrix: numpy array (size of current layer, size of previous layer)
    b -- bias vector , (size of current layer, 1)
    activation -- activation use in the layer, "sigmoid" , "relu"

    Return:
    A -- output of the activation fucntion
    cache -- python tuple that contain "linear_cache" and activation_cache"
            stored for computing backward efficiently
    """
    if activation == "sigmoid":
        Z, linear_cache = linear_forward(A_prev, W,b)
        A, activation_cache = sigmoid(Z)

    elif activation == "relu":
        Z, linear_cache = linear_forward(A_prev, W, b)
        A, activation_cache = relu(Z)

    cache = (linear_cache, activation_cache)

    return A, cache


In [52]:
def L_model_forward(X, parameters):
    """
    implement forward propagation for the [Linear -> Relu] * (l-1) -> LINEAR -> sigmoid computation

    Arguments:
    X -- data, (input size, no of examples)
    parameters -- output of initialize_parameters_deep()

    Return:
    AL - activation value from the last layer
    cache -- list of caches containing ever cache from linear_Activation_forward() from  0 to L-1
    """
    cache = []
    A =X
    L = len(parameters) // 2 #no of layers in the neural netwrok

    for l in range(1, L):
        A_prev = A
        A, cache = linear_activation_forward(A_prev, parameters['W' + str(l)], parameters['b' + str(l)], 'relu')
        caches.append(cache)
    
    AL, cache = linear_activation_forward(A, parameters['W' + str(L)], parameters['b' + str(L)], 'sigmoid')
    cache.append(cache)

    return AL, caches